# FASE 3 — Asset Health Index
## Formula: 40% Temp + 40% Vibration + 20% Current

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-darkgrid')
df = pd.read_csv('../data/raw_data.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [ ]:
# Normalize globally
df['temp_norm'] = 1 - (df['temperature_c'] - df['temperature_c'].min()) / (df['temperature_c'].max() - df['temperature_c'].min())
df['vib_norm']  = 1 - (df['vibration_mm_s'] - df['vibration_mm_s'].min()) / (df['vibration_mm_s'].max() - df['vibration_mm_s'].min())
df['cur_norm']  = 1 - (df['current_a'] - df['current_a'].min()) / (df['current_a'].max() - df['current_a'].min())
df['health_score'] = (0.40*df['temp_norm'] + 0.40*df['vib_norm'] + 0.20*df['cur_norm']) * 100
print('Health Score Statistics:')
print(df['health_score'].describe())

In [ ]:
def categorize(score):
    if score >= 80: return 'HEALTHY'
    elif score >= 60: return 'WARNING'
    else: return 'CRITICAL'
df['health_cat'] = df['health_score'].apply(categorize)
health_by_machine = df.groupby('machine_id')['health_score'].mean()
print('\nHealth Score per Machine:')
for m, sc in health_by_machine.items():
    cat = categorize(sc)
    print(f'{m}: {sc:.1f}% — {cat}')

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
cats = health_by_machine.apply(categorize)
colors_map = {'HEALTHY':'#4CAF50','WARNING':'#FF9800','CRITICAL':'#F44336'}
bars = axes[0].bar(health_by_machine.index, health_by_machine.values,
                   color=[colors_map[cats[m]] for m in health_by_machine.index])
axes[0].axhline(80, ls='--', color='#4CAF50', linewidth=1.5, label='Healthy (80)')
axes[0].axhline(60, ls='--', color='#FF9800', linewidth=1.5, label='Warning (60)')
axes[0].set_ylim(0,110)
axes[0].set_title('Health Score per Machine', fontweight='bold')
axes[0].set_ylabel('Health Score (%)')
for b,m in zip(bars,health_by_machine.index):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{health_by_machine[m]:.1f}%\n{cats[m]}',
                 ha='center', fontsize=8.5, fontweight='bold')
axes[0].legend()
cat_counts = df['health_cat'].value_counts()
axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
            colors=['#FF9800','#4CAF50'], startangle=90)
axes[1].set_title('Distribution of Health Categories', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plots/health_detail.png', dpi=150)
plt.show()

## Health Score Summary

| Machine | Health Score | Kategori |
|---------|-------------|----------|
| MOTOR_01 | 63.7% | WARNING |
| MOTOR_02 | 64.2% | WARNING |
| MOTOR_03 | 65.4% | WARNING |
| PUMP_01 | 65.0% | WARNING |
| COMPRESSOR_01 | 63.9% | WARNING |